In [0]:
dbutils.widgets.dropdown(name = "environment", defaultValue= "dev",choices= ["dev","prd","qa"],label = "select Environment")
env = dbutils.widgets.get("environment")
print(env)

catalog = f"saleslake_{env}"
bronzeSchema = f"bronze_schema_{env}"

targetTable = f"{catalog}.{bronzeSchema}.rawDiscount"
# print(tablName)
# srcFileLoc=f"s3://saleslake-748005667258-eu-north-1-an/saleslake/{env}/source_files/discount/"
srcFileLoc=f"/Volumes/{catalog}/{bronzeSchema}/vol_saleslake_src_files_{env}/manual_discount/"
print(f"Target table : {targetTable}")
print(f"Source files : {srcFileLoc}")

In [0]:
spark.sql(f"""
COPY INTO {targetTable}
FROM (
    SELECT discount_id,discount_code,discount_name,discount_type,discount_value,min_purchase_amount AS min_amount,max_discount_amount AS max_amount,valid_from,valid_to,applicable_segment,applicable_category,usage_limit_per_customer AS usage_limit,is_active,created_date,current_timestamp AS ingest_ts
    FROM '{srcFileLoc}'
    )
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true')
COPY_OPTIONS ('mergeSchema' = 'false')
""")
print("Discount Bronze Load Successful")
print(f"Bronze load complete for {targetTable}")

In [0]:
spark.sql(f"""
        SELECT COUNT(*) AS total_records
        FROM {targetTable}
    """).show()